# World Anvil Sync Notebook

This notebook syncs all content from Google Spreadsheets to World Anvil using the **internal web API**.

## Setup Instructions

### Authentication (Browser Session)

This notebook uses your browser session cookies for authentication - no API keys needed!

#### How to get your cookies:

1. **Log in to World Anvil** in your browser
2. **Open Developer Tools** (F12 or Right-click → Inspect)
3. **Go to the Network tab** and refresh the page
4. **Find any API request** to `worldanvil.com`
5. **Copy the Cookie header** - it will look like:
   ```
   PHPSESSID=xxxxx; REMEMBERME=xxxxx; cf_clearance=xxxxx
   ```
6. **Paste it** in the `WA_COOKIES` field below

**Note**: Your session cookies will expire eventually. If sync stops working, just grab fresh cookies from your browser.

### API Endpoint
- **Base URL**: `https://www.worldanvil.com/api/internal/aboleth`
- **Type**: Internal web API (cookie-based authentication)
- **World**: Orimond

## 1. Setup and Imports

In [ ]:
import pandas as pd
import requests
import json
import time
from typing import Dict, List, Optional
from datetime import datetime

# Import your existing content modules
import FiveETools.fantasy_spells
import FiveETools.fantasy_monster
import FiveETools.fantasy_species
import FiveETools.dieties
import FiveETools.fantasy_diseases
import FiveETools.conditions
import FiveETools.fantasy_languages
import FiveETools.fantasy_magic_items

from world_anvil.helpers.WorldAnvilAPI import WorldAnvilAPI

print("✓ Imports successful")

## 2. Authentication Configuration

**Using Browser Cookies**: Just copy your session cookies from your browser's Developer Tools.

See instructions above for how to extract cookies from your browser.

In [ ]:
# World Anvil Configuration
WA_BASE_URL = "https://www.worldanvil.com"
WA_API_BASE = "https://www.worldanvil.com/api/internal/aboleth"  # Internal web API
WA_WORLD_SLUG = "orimond-gogluness"  # Your world slug (with username)
WA_WORLD_ID = "ff8434dd-4881-488d-bf71-8766c5851a12"  # Your world ID from the payload example

# ============================================
# CATEGORY/FOLDER ID MAPPING
# ============================================
# World Anvil uses category UUIDs to organize articles
# You need to find these IDs for your world's categories

WA_CATEGORY_IDS = {
    "species": "39dd77b9-a867-4b8f-aba5-4448233ec625",
    "spells": "50530b1d-13c2-4095-90c8-d95250f90846", 
    "monsters": "690399f4-1884-45c2-8f2f-19ea0a6efb32",
    "dieties": "ce8235d2-46bd-42c7-b66f-3ffde3d8ee9f",
    "diseases": "2dadd34e-b1bf-4f79-8744-c7b40f6b0c67",
    "conditions": "a8004c04-696c-4291-b8b3-382b92af9e46",
    "languages": "be600c3a-1846-4bf6-ae7c-5dac0508638d",
    "magic_items": "1b530865-8ac2-4332-8176-d05bf025bfa9"
}

# How to find category IDs:
# 1. Go to World Anvil and navigate to a category
# 2. Open DevTools → Network tab
# 3. Look at the URL or API requests - the category ID will be in the URL
# Example: /p/athena/category/39dd77b9-a867-4b8f-aba5-4448233ec625/edit

# ============================================
# AUTHENTICATION (Browser Cookies)
# ============================================

# Paste your full Cookie header from browser DevTools here:
WA_COOKIES = "cf_clearance=t6tTiv_ADkVs6FqmKt5ibuCur90f6hV2Qsgde7.v5UM-1766422617-1.2.1.1-wQKxsDfpVOk7PH_naQkxyn6Aoyv8LM4D2A4QxBDIpbypYI_PCbbBNotQuhXyverxYxkfK6xOSeXHqpIdymphU3ZsZo3wJ8McP76SzKP0nKlEOkH4MWg8TJeRh9XCoOySX4.2iu95f1MURO9Wpjtgcf36hNw16hE99FDoouKcHJXjx270rf2nl.t6qFFuZfqkamcMmoFjL2jt3.TITFK.0sRldSmC3ZzqlCFgDF9XV6M; PHPSESSID=ei2aouubdhclbm2fffi4cvrkl9; REMEMBERME=QXBwQnVuZGxlXEVudGl0eVxVc2VyOlIyOW5iSFZ1WlhOejoxNzY3MDI2NzA4OmYyNjJlNmVkOThjZDgzY2UxMmM5Y2QyZDdlNGZjYzJiMjlhMDYwNzYyMDBmY2Y4NGYwOWE4MTNlNzcwMTA1ZTM%3D; _gcl_au=1.1.1477590654.1763815354"

# User-Agent from your browser (optional but recommended)
WA_USER_AGENT = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Safari/605.1.15"

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if WA_COOKIES:
    # Use cookie-based authentication (browser session)
    session.headers.update({
        "Cookie": WA_COOKIES,
        "Accept": "application/json",
        "Accept-Language": "en-CA,en-US;q=0.9,en;q=0.8",
        "Content-Type": "application/json",
        "User-Agent": WA_USER_AGENT,
        "Referer": f"{WA_BASE_URL}/w/{WA_WORLD_SLUG}",
        "Origin": WA_BASE_URL,
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-origin"
    })
    print("✓ Using browser cookie authentication")
    print(f"✓ API Base: {WA_API_BASE}")
    print(f"✓ World: {WA_WORLD_SLUG}")
    print(f"✓ World ID: {WA_WORLD_ID}")
    print(f"\n✓ Category IDs configured:")
    for category, cat_id in WA_CATEGORY_IDS.items():
        status = "✓" if cat_id else "⚠️ "
        print(f"  {status} {category}: {cat_id or 'Not configured'}")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set WA_COOKIES with your browser session cookies")

## 3. Load All Content from Spreadsheets

In [ ]:
# Load all content from existing modules
print("Loading content from spreadsheets...")

content_data = {
    "spells": FiveETools.fantasy.spells.spells_list,

}

# Print summary
for content_type, items in content_data.items():
    print(f"✓ Loaded {len(items)} {content_type}")

total_items = sum(len(items) for items in content_data.values())
print(f"\n✓ Total items to sync: {total_items}")

## 4. World Anvil API Helpers

In [ ]:
# Initialize API helper
wa_api = WorldAnvilAPI(session, WA_WORLD_ID, WA_WORLD_SLUG, WA_API_BASE)
print("✓ World Anvil API helper initialized")

## 5.2 Legacy Content Converters

Convert 5etools format to World Anvil format (kept for backward compatibility)

## 5. Mapping-Based Converters

Load mapping files from `world_anvil/mapping/` to convert spreadsheet data to World Anvil format.

### Mapping File Format

Mapping files use JSON with World Anvil field names as keys and Google Sheets column headers as values:

```json
{
  "title": "Name",
  "content": "Description",
  "content.Intro": "Intro",
  "traits[0].name": "Trait 1",
  "speed.walk": "Walk Speed"
}
```

Supports:
- Simple fields: `"title": "Name"`
- Nested objects: `"speed.walk": "Walk Speed"`
- Arrays: `"traits[0].name": "Trait 1"`
- Content sections: `"content.Section Name": "Column"`

In [ ]:
import os
from pathlib import Path
import re

# Mapping directory
MAPPING_DIR = "world_anvil/mapping"

def load_mapping(mapping_name: str) -> Dict:
    """Load a mapping file from the mapping directory"""
    mapping_path = os.path.join(MAPPING_DIR, f"{mapping_name}.json")
    
    if not os.path.exists(mapping_path):
        print(f"⚠️  Mapping file not found: {mapping_path}")
        return {}
    
    with open(mapping_path, 'r') as f:
        mapping = json.load(f)
    
    return mapping

def set_nested_value(obj: Dict, path: str, value):
    """Set a value in a nested dictionary using dot notation
    
    Examples:
        set_nested_value(obj, "speed.walk", 30) -> obj["speed"]["walk"] = 30
        set_nested_value(obj, "traits[0].name", "Darkvision") -> obj["traits"][0]["name"] = "Darkvision"
        set_nested_value(obj, "content.Intro", "text") -> Appends to content field
    """
    # Handle content.Section format specially
    if path.startswith("content."):
        section_name = path.split(".", 1)[1]
        if "content" not in obj:
            obj["content"] = ""
        
        # Append section with heading
        if obj["content"]:
            obj["content"] += "\n\n"
        obj["content"] += f"[h2]{section_name}[/h2]\n{text_to_bbcode(str(value))}"
        return
    
    # Parse array indices like "traits[0].name"
    parts = []
    for part in path.split("."):
        match = re.match(r"([^\[]+)\[(\d+)\]", part)
        if match:
            parts.append((match.group(1), int(match.group(2))))
        else:
            parts.append((part, None))
    
    # Navigate to the target
    current = obj
    for i, (key, index) in enumerate(parts[:-1]):
        if index is not None:
            # Array access
            if key not in current:
                current[key] = []
            # Extend array if needed
            while len(current[key]) <= index:
                current[key].append({})
            current = current[key][index]
        else:
            # Object access
            if key not in current:
                current[key] = {}
            current = current[key]
    
    # Set the final value
    final_key, final_index = parts[-1]
    if final_index is not None:
        if final_key not in current:
            current[final_key] = []
        while len(current[final_key]) <= final_index:
            current[final_key].append(None)
        current[final_key][final_index] = value
    else:
        current[final_key] = value

def convert_with_mapping(row: Dict, mapping: Dict, template_type: str = "article") -> Dict:
    """Convert a spreadsheet row to World Anvil format using a mapping
    
    Args:
        row: Dictionary with spreadsheet column headers as keys
        mapping: Mapping from WA fields to spreadsheet columns
        template_type: World Anvil template type
    
    Returns:
        Dictionary in World Anvil API format
    """
    result = {
        "templateType": template_type,
        "state": "private",
        "isWip": False,
        "isDraft": False,
        "content": ""
    }
    
    # Apply mapping
    for wa_field, sheet_column in mapping.items():
        if not sheet_column or sheet_column not in row:
            continue
        
        value = row[sheet_column]
        
        # Skip empty values
        if pd.isna(value) or (isinstance(value, str) and not value.strip()):
            continue
        
        # Handle special fields
        if wa_field == "tags" and isinstance(value, str):
            # Tags should be comma-separated string
            value = value if "," in value else value.replace(" ", ",")
        
        # Set the value using dot notation
        set_nested_value(result, wa_field, value)
    
    # Ensure title exists
    if "title" not in result:
        result["title"] = "Untitled"
    
    # Convert content to BBCode if it's plain text
    if result.get("content") and not result["content"].startswith("["):
        result["content"] = text_to_bbcode(result["content"])
    
    return result

# Load all mapping files
MAPPINGS = {}

mapping_files = {
    "fantasy_spells": "gspread_fantasy_spells_2_WA",
    "fantasy_species": "gspread_fantasy_species_2_WA",
    "fantasy_monsters": "gspread_fantasy_monsters_2_wa",
}

for content_type, mapping_file in mapping_files.items():
    mapping = load_mapping(mapping_file)
    if mapping:
        MAPPINGS[content_type] = mapping
        print(f"✓ Loaded mapping: {content_type} ({len(mapping)} fields)")
    else:
        print(f"⚠️  No mapping loaded: {content_type}")

print(f"\n✓ Mapping system ready with {len(MAPPINGS)} mappings")

In [ ]:
def text_to_bbcode(text: str) -> str:
    """Convert plain text to BBCode format with paragraphs"""
    if not text:
        return ""
    
    # Split by double newlines for paragraphs
    paragraphs = text.split("\n\n")
    bbcode_paragraphs = [f"[p]{p.strip()}[/p]" for p in paragraphs if p.strip()]
    return "\n".join(bbcode_paragraphs)

def convert_spell_to_wa(spell: Dict) -> Dict:
    """Convert spell data to World Anvil article format"""
    # Build description from entries
    description_parts = []
    for entry in spell.get("entries", []):
        if isinstance(entry, str):
            description_parts.append(entry)
        else:
            description_parts.append(json.dumps(entry))
    
    # Add higher level info
    if "entriesHigherLevel" in spell:
        higher_entries = spell["entriesHigherLevel"][0].get("entries", [])
        if higher_entries:
            description_parts.append("**At Higher Levels:** " + " ".join(higher_entries))
    
    # Build metadata
    metadata_parts = [
        f"**Level:** {spell.get('level', 0)}",
        f"**School:** {spell.get('school', 'N/A')}",
        f"**Casting Time:** {spell.get('time', [{}])[0].get('number', 1)} {spell.get('time', [{}])[0].get('unit', 'action')}",
        f"**Range:** {spell.get('range', {}).get('distance', {}).get('type', 'self')}",
        f"**Components:** {'V' if spell.get('components', {}).get('v') else ''}{'S' if spell.get('components', {}).get('s') else ''}{'M' if spell.get('components', {}).get('m') else ''}",
        f"**Duration:** {spell.get('duration', [{}])[0].get('type', 'instantaneous')}",
    ]
    
    full_content = "\n\n".join(metadata_parts + description_parts)
    
    result = {
        "title": spell.get("name", "Unnamed Spell"),
        "content": text_to_bbcode(full_content),
        "template": "spell",
        "templateType": "spell",
        "tags": spell.get("miscTags", []),
        "state": "private",
        "isWip": False,
        "isDraft": False
    }
    
    # Add category/folder ID if configured
    if WA_CATEGORY_IDS.get("spells"):
        result["folderId"] = WA_CATEGORY_IDS["spells"]
    
    return result

def convert_monster_to_wa(monster: Dict) -> Dict:
    """Convert monster data to World Anvil article format"""
    # Build stat block
    stats_parts = [
        f"**Size:** {monster.get('size', ['M'])[0]}",
        f"**Type:** {monster.get('type', 'humanoid')}",
        f"**Alignment:** {monster.get('alignment', ['N'])[0]}",
        f"**AC:** {monster.get('ac', [{}])[0].get('ac', 10)}",
        f"**HP:** {monster.get('hp', {}).get('average', 0)}",
        f"**Speed:** {monster.get('speed', {}).get('walk', 30)}ft",
        f"**CR:** {monster.get('cr', '0')}",
        "",
        f"**STR** {monster.get('str', 10)} | **DEX** {monster.get('dex', 10)} | **CON** {monster.get('con', 10)} | **INT** {monster.get('int', 10)} | **WIS** {monster.get('wis', 10)} | **CHA** {monster.get('cha', 10)}",
    ]
    
    description = ""
    if monster.get("fluff"):
        fluff_entries = monster["fluff"].get("entries", [])
        if fluff_entries:
            description = fluff_entries[0] if isinstance(fluff_entries[0], str) else ""
    
    full_content = "\n\n".join(stats_parts + ([description] if description else []))
    
    result = {
        "title": monster.get("name", "Unnamed Monster"),
        "content": text_to_bbcode(full_content),
        "template": "article",  # Generic article for monsters
        "templateType": "article",
        "state": "private",
        "isWip": False,
        "isDraft": False
    }
    
    # Add category/folder ID if configured
    if WA_CATEGORY_IDS.get("monsters"):
        result["folderId"] = WA_CATEGORY_IDS["monsters"]
    
    return result

def convert_species_to_wa(species: Dict) -> Dict:
    """Convert species data to World Anvil article format"""
    # Extract traits
    trait_parts = []
    for entry in species.get("entries", []):
        if isinstance(entry, dict) and entry.get("type") == "entries":
            name = entry.get("name", "")
            content = " ".join([e if isinstance(e, str) else str(e) for e in entry.get("entries", [])])
            trait_parts.append(f"**{name}:** {content}")
    
    description_parts = []
    fluff = species.get("fluff", {})
    if fluff:
        for entry in fluff.get("entries", []):
            if isinstance(entry, dict):
                name = entry.get('name', '')
                content_entries = entry.get("entries", [])
                desc_text = " ".join([str(e) for e in content_entries])
                if name:
                    description_parts.append(f"**{name}**\n{desc_text}")
                else:
                    description_parts.append(desc_text)
            elif isinstance(entry, str):
                description_parts.append(entry)
    
    full_content = "\n\n".join(trait_parts + description_parts)
    
    result = {
        "title": species.get("name", "Unnamed Species"),
        "content": text_to_bbcode(full_content),
        "template": "species",
        "templateType": "species",
        "state": "private",
        "isWip": False,
        "isDraft": False
    }
    
    # Add category/folder ID if configured
    if WA_CATEGORY_IDS.get("species"):
        result["folderId"] = WA_CATEGORY_IDS["species"]
    
    return result

def convert_generic_to_wa(item: Dict, category: str, template_type: str = "article") -> Dict:
    """Generic converter for other content types"""
    content = ""
    
    # Try to extract description/entries
    if "entries" in item:
        entries = item["entries"]
        if isinstance(entries, list):
            content = "\n\n".join([str(e) for e in entries])
        else:
            content = str(entries)
    elif "description" in item:
        content = item["description"]
    
    result = {
        "title": item.get("name", f"Unnamed {category}"),
        "content": text_to_bbcode(content),
        "template": template_type,
        "templateType": template_type,
        "state": "private",
        "isWip": False,
        "isDraft": False
    }
    
    # Add category/folder ID if configured
    if WA_CATEGORY_IDS.get(category):
        result["folderId"] = WA_CATEGORY_IDS[category]
    
    return result

# Converter mapping
CONVERTERS = {
    "spells": convert_spell_to_wa,
    "monsters": convert_monster_to_wa,
    "species": convert_species_to_wa,
    "dieties": lambda x: convert_generic_to_wa(x, "dieties", "article"),
    "diseases": lambda x: convert_generic_to_wa(x, "diseases", "condition"),
    "conditions": lambda x: convert_generic_to_wa(x, "conditions", "condition"),
    "languages": lambda x: convert_generic_to_wa(x, "languages", "language"),
}

print("✓ Content converters ready")

In [ ]:
# Create converters using mappings
def create_mapping_converter(mapping: Dict, template_type: str):
    """Create a converter function from a mapping"""
    def converter(row: Dict) -> Dict:
        result = convert_with_mapping(row, mapping, template_type)
        return result
    return converter

# Add mapping-based converters to CONVERTERS dict
if "fantasy_spells" in MAPPINGS:
    CONVERTERS["fantasy_spells"] = create_mapping_converter(MAPPINGS["fantasy_spells"], "spell")
    print("✓ Created fantasy_spells converter from mapping")

if "fantasy_species" in MAPPINGS:
    CONVERTERS["fantasy_species"] = create_mapping_converter(MAPPINGS["fantasy_species"], "species")
    print("✓ Created fantasy_species converter from mapping")

if "fantasy_monsters" in MAPPINGS:
    CONVERTERS["fantasy_monsters"] = create_mapping_converter(MAPPINGS["fantasy_monsters"], "article")
    print("✓ Created fantasy_monsters converter from mapping")

print(f"\n✓ Total converters available: {len(CONVERTERS)}")
print(f"  Mapping-based: {sum(1 for k in CONVERTERS if 'fantasy' in k)}")
print(f"  Legacy: {len(CONVERTERS) - sum(1 for k in CONVERTERS if 'fantasy' in k)}")

## 6. Sync Functions

In [ ]:
def sync_item(item: Dict, content_type: str, converter, category_articles: List[Dict]) -> Dict:
    """Sync a single item to World Anvil
    
    Args:
        item: Item to sync
        content_type: Type of content (spells, species, etc.)
        converter: Converter function
        category_articles: Pre-fetched list of articles in the category
    """
    try:
        # Convert to WA format
        wa_data = converter(item)
        title = wa_data["title"]
        
        # Search in pre-fetched category articles
        existing = None
        title_lower = title.lower()
        for article in category_articles:
            if article.get("title", "").lower() == title_lower:
                existing = article
                break
        
        if existing:
            # Update existing
            result = wa_api.update_article(existing["id"], wa_data)
            return {
                "title": title,
                "action": "updated",
                "success": result is not None,
                "error": None if result else wa_api.last_error or "Update failed"
            }
        else:
            # Create new
            result = wa_api.create_article(wa_data)
            return {
                "title": title,
                "action": "created",
                "success": result is not None,
                "error": None if result else wa_api.last_error or "Creation failed"
            }
    except Exception as e:
        return {
            "title": item.get("name", "Unknown"),
            "action": "error",
            "success": False,
            "error": str(e)
        }

def sync_content_type(content_type: str, items: List[Dict], batch_size: int = 10, delay: float = 1.0):
    """Sync all items of a content type"""
    print(f"\n{'='*60}")
    print(f"Syncing {len(items)} {content_type}...")
    
    # Check if category ID is configured
    category_id = WA_CATEGORY_IDS.get(content_type)
    if not category_id:
        print(f"⚠️  No category ID configured for {content_type} - skipping")
        return {"created": 0, "updated": 0, "errors": 0, "details": []}
    
    print(f"Using category: {category_id}")
    print(f"{'='*60}")
    
    converter = CONVERTERS.get(content_type)
    if not converter:
        print(f"⚠️  No converter found for {content_type}")
        return {"created": 0, "updated": 0, "errors": 0, "details": []}
    
    # IMPORTANT: Fetch all articles in category ONCE at the start
    print(f"Fetching existing articles from World Anvil...")
    category_articles = wa_api.get_articles_by_category(category_id)
    print(f"Found {len(category_articles)} existing articles in category\n")
    
    results = {
        "created": 0,
        "updated": 0,
        "errors": 0,
        "details": []
    }
    
    for i, item in enumerate(items, 1):
        result = sync_item(item, content_type, converter, category_articles)
        results["details"].append(result)
        
        if result["success"]:
            if result["action"] == "created":
                results["created"] += 1
                print(f"✓ [{i}/{len(items)}] Created: {result['title']}")
                # Add to category_articles so subsequent items can find it
                category_articles.append({"title": result['title']})
            elif result["action"] == "updated":
                results["updated"] += 1
                print(f"↻ [{i}/{len(items)}] Updated: {result['title']}")
        else:
            results["errors"] += 1
            print(f"✗ [{i}/{len(items)}] Error: {result['title']} - {result['error']}")
        
        # Rate limiting
        if i % batch_size == 0:
            print(f"  ... Waiting {delay}s to avoid rate limits ...")
            time.sleep(delay)
    
    # Summary
    print(f"\n{'='*60}")
    print(f"Summary for {content_type}:")
    print(f"  Created: {results['created']}")
    print(f"  Updated: {results['updated']}")
    print(f"  Errors:  {results['errors']}")
    print(f"{'='*60}\n")
    
    return results

print("✓ Sync functions ready (using folder lookup only)")

## 8. Run Full Sync

Sync all content types to World Anvil

In [ ]:
# Run full sync
print("\n" + "="*60)
print("STARTING FULL SYNC TO WORLD ANVIL")
print("="*60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n")

all_results = {}

# Sync each content type
for content_type, items in content_data.items():
    if items:  # Only sync if there are items
        results = sync_content_type(content_type, items, batch_size=10, delay=1.0)
        all_results[content_type] = results
        time.sleep(2)  # Extra delay between content types

# Final summary
print("\n" + "="*60)
print("SYNC COMPLETE")
print("="*60)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nOverall Summary:")

total_created = sum(r["created"] for r in all_results.values())
total_updated = sum(r["updated"] for r in all_results.values())
total_errors = sum(r["errors"] for r in all_results.values())

print(f"  Total Created: {total_created}")
print(f"  Total Updated: {total_updated}")
print(f"  Total Errors:  {total_errors}")
print("="*60)

## 9. Obsidian Portal Import

Import content from Obsidian Portal campaign XML exports (adventure logs, characters, items, wiki pages).

**Prerequisites**:
- Export your Obsidian Portal campaign to XML
- Place the `rocko.xml` file (or your campaign's XML file) in `extracted/op/` directory
- Configure category IDs for the content types you want to import

**XML Structure**:
Obsidian Portal exports use XML format with the following structure:
- `<blog><entry>` - Adventure log entries
- `<wiki><page>` - Wiki pages
- `<characters><character>` - Characters (PCs and NPCs)
- `<items><item>` - Items and equipment

Each element includes:
- `gm_only` attribute - whether content is GM-only
- `<title>` - Name of the entry
- `<content format="html">` - HTML content
- `<published>` - Publication date

In [24]:
# ============================================
# Obsidian Portal Configuration
# ============================================

# Path to Obsidian Portal XML export file
OP_EXPORT_FILE = "extracted/op/rocko.xml"

# Category IDs for different content types
OP_CATEGORY_IDS = {
    "journal": "a8ed8428-2241-43ed-8385-2d652793139a",  # Adventure logs (blog entries)
    "characters": "3d327b14-b7d7-46ea-89ef-0d1b5a6468a9",  # PC/NPC characters
    "items": "1b530865-8ac2-4332-8176-d05bf025bfa9",  # Magic items, equipment
    "wiki": ""  # General wiki pages
}

# Skip GM-only content?
SKIP_GM_ONLY = False

print("Obsidian Portal Configuration:")
print(f"  Export file: {OP_EXPORT_FILE}")
print(f"  Skip GM-only: {SKIP_GM_ONLY}")
print(f"\n  Configured categories:")
for content_type, cat_id in OP_CATEGORY_IDS.items():
    status = "✓" if cat_id else "⚠️ "
    print(f"    {status} {content_type}: {cat_id or 'Not configured'}")

Obsidian Portal Configuration:
  Export file: extracted/op/rocko.xml
  Skip GM-only: False

  Configured categories:
    ✓ journal: a8ed8428-2241-43ed-8385-2d652793139a
    ✓ characters: 3d327b14-b7d7-46ea-89ef-0d1b5a6468a9
    ✓ items: 1b530865-8ac2-4332-8176-d05bf025bfa9
    ⚠️  wiki: Not configured


In [25]:
import os
import xml.etree.ElementTree as ET

def parse_obsidian_portal_xml(filepath):
    """Parse Obsidian Portal XML export and extract content by type
    
    XML structure:
    <campaign>
      <blog>
        <entry gm_only="true|false">
          <title>...</title>
          <content format="html">...</content>
          <published>...</published>
        </entry>
      </blog>
      <wiki>
        <page gm_only="true|false">
          <title>...</title>
          <content format="html">...</content>
        </page>
      </wiki>
      <characters>
        <character gm_only="true|false">
          <title>...</title>
          <content format="html">...</content>
        </character>
      </characters>
      <items>
        <item gm_only="true|false">
          <title>...</title>
          <content format="html">...</content>
        </item>
      </items>
    </campaign>
    """
    if not os.path.exists(filepath):
        print(f"⚠️  File not found: {filepath}")
        return {}
    
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        op_data = {
            "journal": [],
            "wiki": [],
            "characters": [],
            "items": []
        }
        
        # Parse blog entries (adventure log)
        blog = root.find('blog')
        if blog is not None:
            for entry in blog.findall('entry'):
                gm_only = entry.get('gm_only', 'false').lower() == 'true'
                
                # Get title
                title_elem = entry.find('title')
                title = title_elem.text if title_elem is not None else "Untitled Entry"
                
                # Get HTML content (prefer HTML over textile)
                content = ""
                for content_elem in entry.findall('content'):
                    if content_elem.get('format') == 'html' and content_elem.get('gm_only', 'false') == 'false':
                        content = content_elem.text or ""
                        break
                
                # Get published date
                published_elem = entry.find('published')
                published = published_elem.text if published_elem is not None else ""
                
                # Get link
                link_elem = entry.find('link')
                link = link_elem.text if link_elem is not None else ""
                
                op_data["journal"].append({
                    "title": title,
                    "content": content,
                    "published": published,
                    "gm_only": gm_only,
                    "link": link
                })
        
        # Parse wiki pages
        wiki = root.find('wiki')
        if wiki is not None:
            for page in wiki.findall('page'):
                gm_only = page.get('gm_only', 'false').lower() == 'true'
                
                title_elem = page.find('title')
                title = title_elem.text if title_elem is not None else "Untitled Page"
                
                content = ""
                for content_elem in page.findall('content'):
                    if content_elem.get('format') == 'html' and content_elem.get('gm_only', 'false') == 'false':
                        content = content_elem.text or ""
                        break
                
                published_elem = page.find('published')
                published = published_elem.text if published_elem is not None else ""
                
                link_elem = page.find('link')
                link = link_elem.text if link_elem is not None else ""
                
                op_data["wiki"].append({
                    "title": title,
                    "content": content,
                    "published": published,
                    "gm_only": gm_only,
                    "link": link
                })
        
        # Parse characters
        characters = root.find('characters')
        if characters is not None:
            for character in characters.findall('character'):
                gm_only = character.get('gm_only', 'false').lower() == 'true'
                
                title_elem = character.find('title')
                title = title_elem.text if title_elem is not None else "Unnamed Character"
                
                content = ""
                for content_elem in character.findall('content'):
                    if content_elem.get('format') == 'html' and content_elem.get('gm_only', 'false') == 'false':
                        content = content_elem.text or ""
                        break
                
                link_elem = character.find('link')
                link = link_elem.text if link_elem is not None else ""
                
                op_data["characters"].append({
                    "title": title,
                    "content": content,
                    "gm_only": gm_only,
                    "link": link
                })
        
        # Parse items
        items = root.find('items')
        if items is not None:
            for item in items.findall('item'):
                gm_only = item.get('gm_only', 'false').lower() == 'true'
                
                title_elem = item.find('title')
                title = title_elem.text if title_elem is not None else "Unnamed Item"
                
                content = ""
                for content_elem in item.findall('content'):
                    if content_elem.get('format') == 'html' and content_elem.get('gm_only', 'false') == 'false':
                        content = content_elem.text or ""
                        break
                
                link_elem = item.find('link')
                link = link_elem.text if link_elem is not None else ""
                
                op_data["items"].append({
                    "title": title,
                    "content": content,
                    "gm_only": gm_only,
                    "link": link
                })
        
        return op_data
        
    except Exception as e:
        print(f"✗ Error parsing XML: {e}")
        import traceback
        traceback.print_exc()
        return {}

# Load and parse Obsidian Portal XML
print(f"Loading Obsidian Portal data from {OP_EXPORT_FILE}...")
op_data = parse_obsidian_portal_xml(OP_EXPORT_FILE)

print("\nLoaded Obsidian Portal data:")
for content_type, items in op_data.items():
    gm_only_count = sum(1 for item in items if item.get('gm_only', False))
    public_count = len(items) - gm_only_count
    print(f"  {content_type}: {len(items)} items ({public_count} public, {gm_only_count} GM-only)")

total_items = sum(len(items) for items in op_data.values())
print(f"\nTotal items: {total_items}")

Loading Obsidian Portal data from extracted/op/rocko.xml...

Loaded Obsidian Portal data:
  journal: 76 items (48 public, 28 GM-only)
  wiki: 35 items (13 public, 22 GM-only)
  characters: 247 items (88 public, 159 GM-only)
  items: 54 items (16 public, 38 GM-only)

Total items: 412


In [27]:
def convert_op_journal_to_wa(item: Dict) -> Dict:
    """Convert Obsidian Portal adventure log entry to World Anvil article"""
    return {
        "title": item.get("title", "Untitled Entry"),
        "content": item.get("content", ""),
        "templateType": "article",
        "state": "private",
        "tags": "adventure-log, obsidian-portal",
        "folderId": OP_CATEGORY_IDS.get("journal", ""),
    }

def convert_op_wiki_to_wa(item: Dict) -> Dict:
    """Convert Obsidian Portal wiki page to World Anvil article"""
    return {
        "title": item.get("title", "Untitled Page"),
        "content": item.get("content", ""),
        "templateType": "article",
        "state": "private",
        "tags": "wiki, obsidian-portal",
        "folderId": OP_CATEGORY_IDS.get("wiki", ""),
    }

def convert_op_character_to_wa(item: Dict) -> Dict:
    """Convert Obsidian Portal character to World Anvil article
    
    Note: Using 'article' templateType instead of 'character'
    because World Anvil's API doesn't accept 'character' as a valid type.
    """
    return {
        "title": item.get("title", "Unnamed Character"),
        "content": item.get("content", ""),
        "templateType": "article",  # Changed from "character" to "article"
        "state": "private",
        "tags": "character, obsidian-portal",
        "folderId": OP_CATEGORY_IDS.get("characters", ""),
    }

def convert_op_item_to_wa(item: Dict) -> Dict:
    """Convert Obsidian Portal item to World Anvil article
    
    Note: Using 'article' templateType for items.
    World Anvil has an 'item' type but it may have specific requirements.
    """
    return {
        "title": item.get("title", "Unnamed Item"),
        "content": item.get("content", ""),
        "templateType": "article",  # Using "article" for compatibility
        "state": "private",
        "tags": "item, obsidian-portal",
        "folderId": OP_CATEGORY_IDS.get("items", ""),
    }

# Map content types to converters
op_converters = {
    "journal": convert_op_journal_to_wa,
    "wiki": convert_op_wiki_to_wa,
    "characters": convert_op_character_to_wa,
    "items": convert_op_item_to_wa
}

print("✓ Obsidian Portal converters loaded")

✓ Obsidian Portal converters loaded


## 10. Sync Obsidian Portal Content

Sync each content type from Obsidian Portal to World Anvil.

**Note**: Make sure the category IDs are configured in Section 9 before running.

In [28]:
# ============================================
# SYNC OBSIDIAN PORTAL CONTENT TYPE
# ============================================

# Choose which content type to sync
content_type = "characters"  # Options: journal, wiki, characters, items

print("=" * 60)
print("OBSIDIAN PORTAL SYNC TO WORLD ANVIL")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Skip GM-only content: {SKIP_GM_ONLY}")
print()

# Get items and configuration
items = op_data.get(content_type, [])
category_id = OP_CATEGORY_IDS.get(content_type, "")
converter = op_converters.get(content_type)

if not items:
    print(f"⚠️  No {content_type} items loaded")
elif not category_id:
    print(f"⚠️  No category ID configured for {content_type}")
    print(f"   Please set OP_CATEGORY_IDS['{content_type}'] in Section 9")
elif not converter:
    print(f"⚠️  No converter found for {content_type}")
else:
    # Filter out GM-only content if needed
    items_for_sync = []
    for item in items:
        # Check gm_only field from XML parser
        if SKIP_GM_ONLY and item.get("gm_only", False):
            continue
        items_for_sync.append(item)
    
    skipped_count = len(items) - len(items_for_sync)
    
    print("=" * 60)
    print(f"Syncing {len(items_for_sync)} {content_type} from Obsidian Portal...")
    if SKIP_GM_ONLY and skipped_count > 0:
        print(f"  (Skipped {skipped_count} GM-only items)")
    print(f"Using category: {category_id}")
    print("=" * 60)
    print()
    
    # Fetch existing articles in category
    print(f"Fetching existing articles from World Anvil...")
    category_articles = wa_api.get_articles_by_category(category_id)
    print(f"Found {len(category_articles)} existing articles in category\n")
    
    # Sync results
    results = {
        "created": 0,
        "updated": 0,
        "errors": 0,
        "details": []
    }
    
    for i, item in enumerate(items_for_sync, 1):
        result = sync_item(item, content_type, converter, category_articles)
        results["details"].append(result)
        
        if result["success"]:
            if result["action"] == "created":
                results["created"] += 1
                # Add to list for subsequent lookups
                category_articles.append({"title": result['title']})
                print(f"✓ [{i}/{len(items_for_sync)}] Created: {result['title']}")
            else:
                results["updated"] += 1
                print(f"↻ [{i}/{len(items_for_sync)}] Updated: {result['title']}")
        else:
            results["errors"] += 1
            print(f"✗ [{i}/{len(items_for_sync)}] Error: {result['title']} - {result['error']}")
        
        # Rate limiting - progress update every 10 items
        if i % 10 == 0:
            time.sleep(1)  # Brief pause every 10 items
    
    print()
    print("=" * 60)
    print("SYNC COMPLETE")
    print("=" * 60)
    print(f"✓ Created: {results['created']}")
    print(f"↻ Updated: {results['updated']}")
    print(f"✗ Errors: {results['errors']}")
    print("=" * 60)
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Save log
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"world_anvil/op_sync_{content_type}_{timestamp}.json"
    
    with open(log_file, 'w') as f:
        json.dump({
            "timestamp": timestamp,
            "content_type": content_type,
            "skip_gm_only": SKIP_GM_ONLY,
            "total_items": len(items),
            "synced_items": len(items_for_sync),
            "created": results["created"],
            "updated": results["updated"],
            "errors": results["errors"],
            "details": results["details"]
        }, f, indent=2)
    
    print(f"\n✓ Log saved to: {log_file}")

OBSIDIAN PORTAL SYNC TO WORLD ANVIL
Start time: 2025-12-23 11:54:25
Skip GM-only content: False

Syncing 247 characters from Obsidian Portal...
Using category: 3d327b14-b7d7-46ea-89ef-0d1b5a6468a9

Fetching existing articles from World Anvil...
Found 0 existing articles in category

✓ [1/247] Created: Todokai
✓ [2/247] Created: Jokorin, Valeria
✓ [3/247] Created: Jokorin, Yvenna
✓ [4/247] Created: Jacobi
✓ [5/247] Created: André
✓ [6/247] Created: Victor (dead)
✓ [7/247] Created: Fatbastard, Kirin
✓ [8/247] Created: Abystos
✓ [9/247] Created: Daguemeraude, Quoben
✓ [10/247] Created: Grimm
✓ [11/247] Created: Portepoisse, Rabadin
✓ [12/247] Created: Brightshoulder, Kinedir
✓ [13/247] Created: Paro Paro
✓ [14/247] Created: Danarha
✓ [15/247] Created: Nebreas
✓ [16/247] Created: Freuhorn
✓ [17/247] Created: Narsim
✓ [18/247] Created: Blackblossom, Rothgui
✓ [19/247] Created: Grey, Duncan
✓ [20/247] Created: Ashcloak, Dwodrec
✓ [21/247] Created: Charringrose, Hédémora
✓ [22/247] Created: F

---

## 11. Delete Duplicates

Find and delete duplicate articles in World Anvil (same title, different IDs)

In [ ]:
def find_duplicates_in_category(category_id: str, category_name: str) -> Dict:
    """Find duplicate articles in a category (same title, different IDs)"""
    print(f"\nSearching for duplicates in {category_name}...")
    
    if not category_id:
        print(f"⚠️  No category ID configured for {category_name}")
        return {"duplicates": [], "by_title": {}}
    
    articles = wa_api.get_articles_by_category(category_id)
    
    # Group by title
    by_title = {}
    for article in articles:
        title = article.get("title", "").lower()
        if title not in by_title:
            by_title[title] = []
        by_title[title].append(article)
    
    # Find duplicates (titles with multiple IDs)
    duplicates = []
    for title, articles_list in by_title.items():
        if len(articles_list) > 1:
            duplicates.append({
                "title": title,
                "count": len(articles_list),
                "articles": articles_list
            })
    
    print(f"  Found {len(articles)} total articles")
    print(f"  Found {len(duplicates)} duplicate titles")
    
    return {
        "duplicates": duplicates,
        "by_title": by_title
    }

def delete_article(article_id: str) -> bool:
    """Delete an article by ID using World Anvil internal API
    
    Based on delete.request.txt:
    DELETE https://www.worldanvil.com/api/internal/aboleth/article?id={id}
    """
    try:
        url = f"{WA_API_BASE}/article"
        params = {"id": article_id}
        
        # DELETE method with ID in query parameter (no payload needed)
        response = wa_api.session.delete(url, params=params, json={}, timeout=30)
        response.raise_for_status()
        
        return True
    except Exception as e:
        print(f"  Delete error: {e}")
        return False

# Find duplicates in all configured categories
all_duplicates = {}

print("=" * 60)
print("SEARCHING FOR DUPLICATES")
print("=" * 60)

for category_name, category_id in WA_CATEGORY_IDS.items():
    if category_id:
        result = find_duplicates_in_category(category_id, category_name)
        if result["duplicates"]:
            all_duplicates[category_name] = result

# Summary
print("\n" + "=" * 60)
print("DUPLICATE SUMMARY")
print("=" * 60)

total_duplicate_titles = sum(len(d["duplicates"]) for d in all_duplicates.values())
total_duplicate_articles = sum(
    sum(dup["count"] - 1 for dup in d["duplicates"])  # Count extras only
    for d in all_duplicates.values()
)

if total_duplicate_titles == 0:
    print("\n✓ No duplicates found!")
else:
    print(f"\nFound {total_duplicate_titles} duplicate titles")
    print(f"Total extra articles to remove: {total_duplicate_articles}\n")
    
    for category_name, data in all_duplicates.items():
        print(f"\n{category_name.upper()}:")
        for dup in data["duplicates"]:
            print(f"  '{dup['title']}' - {dup['count']} copies")
            for i, article in enumerate(dup["articles"], 1):
                date = article.get('updateDate', {}).get('date', 'unknown')
                print(f"    [{i}] ID: {article['id'][:8]}... (updated: {date})")

print("\n" + "=" * 60)

### 11.1 Delete Duplicate Articles

**WARNING**: This will permanently delete articles. Review the list above carefully!

**Strategy**: Keep the most recently updated article, delete older duplicates.

In [ ]:
# Configuration
DRY_RUN = False  # Set to False to actually delete
DELETE_OLDER_DUPLICATES = True  # Keep most recent, delete older

print("=" * 60)
print(f"DELETING DUPLICATES {'(DRY RUN)' if DRY_RUN else '(LIVE - DESTRUCTIVE)'}")
print("=" * 60)

if DRY_RUN:
    print("\n⚠️  DRY RUN MODE: No articles will be deleted")
    print("   Set DRY_RUN = False to perform actual deletion\n")
else:
    print("\n🚨 LIVE MODE: Articles WILL BE DELETED")
    print("   This action cannot be undone!\n")

deletion_results = {
    "deleted": 0,
    "errors": 0,
    "skipped": 0,
    "details": []
}

for category_name, data in all_duplicates.items():
    print(f"\n{'='*60}")
    print(f"Processing {category_name}")
    print(f"{'='*60}")
    
    for dup in data["duplicates"]:
        title = dup["title"]
        articles = dup["articles"]
        
        print(f"\n'{title}' ({len(articles)} copies)")
        
        if DELETE_OLDER_DUPLICATES:
            # Sort by update date (most recent first)
            sorted_articles = sorted(
                articles,
                key=lambda a: a.get('updateDate', {}).get('date', ''),
                reverse=True
            )
            
            # Keep first (most recent), delete rest
            keep_article = sorted_articles[0]
            delete_articles = sorted_articles[1:]
            
            keep_date = keep_article.get('updateDate', {}).get('date', 'unknown')
            print(f"  KEEP: {keep_article['id'][:8]}... (updated: {keep_date})")
            
            for article in delete_articles:
                article_id = article['id']
                date = article.get('updateDate', {}).get('date', 'unknown')
                
                if DRY_RUN:
                    print(f"  [DRY RUN] Would delete: {article_id[:8]}... (updated: {date})")
                    deletion_results["skipped"] += 1
                else:
                    print(f"  Deleting: {article_id[:8]}... (updated: {date})")
                    success = delete_article(article_id)
                    
                    if success:
                        print(f"    ✓ Deleted")
                        deletion_results["deleted"] += 1
                    else:
                        print(f"    ✗ Failed to delete")
                        deletion_results["errors"] += 1
                    
                    time.sleep(0.5)  # Rate limiting
                
                deletion_results["details"].append({
                    "title": title,
                    "id": article_id,
                    "date": date,
                    "action": "deleted" if not DRY_RUN else "dry-run"
                })

# Summary
print("\n" + "=" * 60)
print("DELETION SUMMARY")
print("=" * 60)

if DRY_RUN:
    print(f"\nDry run complete:")
    print(f"  Would delete: {deletion_results['skipped']} articles")
else:
    print(f"\nDeletion complete:")
    print(f"  Deleted: {deletion_results['deleted']} articles")
    print(f"  Errors: {deletion_results['errors']} articles")

# Save log
deletion_log_file = f"world_anvil_duplicate_deletion_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(deletion_log_file, "w", encoding="utf-8") as f:
    json.dump(deletion_results, f, indent=2, ensure_ascii=False)

print(f"\n✓ Deletion log saved to: {deletion_log_file}")
print("=" * 60)

---

## 12. Clean Up Orphaned Articles

Delete articles in World Anvil that are NOT present in the spreadsheet

In [ ]:
def find_orphaned_articles(category_id: str, category_name: str, spreadsheet_items: List[Dict]) -> List[Dict]:
    """Find articles in WA that are not in the spreadsheet"""
    print(f"\nSearching for orphaned articles in {category_name}...")
    
    if not category_id:
        print(f"⚠️  No category ID configured for {category_name}")
        return []
    
    # Get all articles from WA
    wa_articles = wa_api.get_articles_by_category(category_id)
    
    # Get all titles from spreadsheet (normalized)
    spreadsheet_titles = set()
    for item in spreadsheet_items:
        title = item.get("name", item.get("title", "")).strip().lower()
        if title:
            spreadsheet_titles.add(title)
    
    # Find WA articles not in spreadsheet
    orphaned = []
    for article in wa_articles:
        wa_title = article.get("title", "").strip().lower()
        if wa_title not in spreadsheet_titles:
            orphaned.append(article)
    
    print(f"  WA articles: {len(wa_articles)}")
    print(f"  Spreadsheet items: {len(spreadsheet_titles)}")
    print(f"  Orphaned (in WA only): {len(orphaned)}")
    
    return orphaned

# Find orphaned articles in all categories
all_orphaned = {}

print("=" * 60)
print("SEARCHING FOR ORPHANED ARTICLES")
print("=" * 60)
print("\nOrphaned = In World Anvil but NOT in spreadsheet\n")

# Map category names to spreadsheet data
category_data_map = {
    "spells": content_data.get("spells", []),
    "monsters": content_data.get("monsters", []),
    "species": content_data.get("species", []),
    "dieties": content_data.get("dieties", []),
    "diseases": content_data.get("diseases", []),
    "conditions": content_data.get("conditions", []),
    "languages": content_data.get("languages", []),
}

for category_name, category_id in WA_CATEGORY_IDS.items():
    if category_id and category_name in category_data_map:
        spreadsheet_items = category_data_map[category_name]
        orphaned = find_orphaned_articles(category_id, category_name, spreadsheet_items)
        
        if orphaned:
            all_orphaned[category_name] = orphaned

# Summary
print("\n" + "=" * 60)
print("ORPHANED ARTICLES SUMMARY")
print("=" * 60)

total_orphaned = sum(len(articles) for articles in all_orphaned.values())

if total_orphaned == 0:
    print("\n✓ No orphaned articles found!")
    print("   All WA articles exist in spreadsheets.")
else:
    print(f"\nFound {total_orphaned} orphaned articles:\n")
    
    for category_name, articles in all_orphaned.items():
        print(f"\n{category_name.upper()} ({len(articles)} orphaned):")
        for article in articles[:20]:  # Show first 20
            title = article.get('title', 'Untitled')
            article_id = article.get('id', 'no-id')
            date = article.get('updateDate', {}).get('date', 'unknown')
            print(f"  - '{title}' (ID: {article_id[:8]}..., updated: {date})")
        
        if len(articles) > 20:
            print(f"  ... and {len(articles) - 20} more")

print("\n" + "=" * 60)

### 12.1 Delete Orphaned Articles

**WARNING**: This will permanently delete articles that exist in World Anvil but NOT in your spreadsheets!

**Use Case**: Clean up after removing items from spreadsheets or after importing from other sources.

**⚠️ IMPORTANT**: Make sure your spreadsheets are complete and up-to-date before running this!

In [ ]:
# Configuration
DRY_RUN_ORPHANED = False  # Set to False to actually delete
CONFIRM_CATEGORY = ""  # Set to category name to enable deletion (e.g., "spells")
                        # Leave empty to prevent accidental deletion

print("=" * 60)
print(f"DELETING ORPHANED ARTICLES {'(DRY RUN)' if DRY_RUN_ORPHANED else '(LIVE - DESTRUCTIVE)'}")
print("=" * 60)

if not CONFIRM_CATEGORY:
    print("\n⚠️  SAFETY LOCK ENABLED")
    print("   Set CONFIRM_CATEGORY to a category name to enable deletion")
    print(f"   Available categories: {list(all_orphaned.keys())}\n")
elif DRY_RUN_ORPHANED:
    print("\n⚠️  DRY RUN MODE: No articles will be deleted")
    print("   Set DRY_RUN_ORPHANED = False to perform actual deletion\n")
else:
    print("\n🚨 LIVE MODE: Articles WILL BE DELETED")
    print("   This action cannot be undone!")
    print(f"   Category: {CONFIRM_CATEGORY}\n")

orphan_deletion_results = {
    "deleted": 0,
    "errors": 0,
    "skipped": 0,
    "details": []
}

if not CONFIRM_CATEGORY:
    print("\n⚠️  No category confirmed. Skipping deletion.")
    print("   Set CONFIRM_CATEGORY = 'category_name' to proceed.")
else:
    # Only process confirmed category
    if CONFIRM_CATEGORY not in all_orphaned:
        print(f"\n⚠️  Category '{CONFIRM_CATEGORY}' not found or has no orphaned articles.")
    else:
        articles = all_orphaned[CONFIRM_CATEGORY]
        
        print(f"\n{'='*60}")
        print(f"Processing {CONFIRM_CATEGORY} ({len(articles)} articles)")
        print(f"{'='*60}\n")
        
        for i, article in enumerate(articles, 1):
            article_id = article['id']
            title = article.get('title', 'Untitled')
            date = article.get('updateDate', {}).get('date', 'unknown')
            
            if DRY_RUN_ORPHANED:
                print(f"[{i}/{len(articles)}] [DRY RUN] Would delete: '{title}'")
                print(f"            ID: {article_id[:8]}..., updated: {date}")
                orphan_deletion_results["skipped"] += 1
            else:
                print(f"[{i}/{len(articles)}] Deleting: '{title}'")
                print(f"            ID: {article_id[:8]}..., updated: {date}")
                
                success = delete_article(article_id)
                
                if success:
                    print(f"            ✓ Deleted")
                    orphan_deletion_results["deleted"] += 1
                else:
                    print(f"            ✗ Failed to delete")
                    orphan_deletion_results["errors"] += 1
                
                time.sleep(0.5)  # Rate limiting
            
            orphan_deletion_results["details"].append({
                "title": title,
                "id": article_id,
                "date": date,
                "category": CONFIRM_CATEGORY,
                "action": "deleted" if not DRY_RUN_ORPHANED else "dry-run"
            })
            
            # Progress indicator every 10 items
            if i % 10 == 0:
                print(f"\n  ... Processed {i}/{len(articles)} ...\n")

# Summary
print("\n" + "=" * 60)
print("ORPHANED DELETION SUMMARY")
print("=" * 60)

if not CONFIRM_CATEGORY:
    print("\n⚠️  No deletion performed (CONFIRM_CATEGORY not set)")
elif DRY_RUN_ORPHANED:
    print(f"\nDry run complete:")
    print(f"  Would delete: {orphan_deletion_results['skipped']} articles")
    print(f"  Category: {CONFIRM_CATEGORY}")
else:
    print(f"\nDeletion complete:")
    print(f"  Deleted: {orphan_deletion_results['deleted']} articles")
    print(f"  Errors: {orphan_deletion_results['errors']} articles")
    print(f"  Category: {CONFIRM_CATEGORY}")

# Save log
if orphan_deletion_results["details"]:
    orphan_log_file = f"world_anvil_orphan_deletion_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(orphan_log_file, "w", encoding="utf-8") as f:
        json.dump(orphan_deletion_results, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ Deletion log saved to: {orphan_log_file}")

print("=" * 60)

## 14. Fix Invalid Author Data

Fix articles with null/invalid author data by setting the correct author ID.

**Purpose**: Repair articles that cause "can't access property id, e.author is null" errors.

**Safety Features**:
- DRY_RUN mode (enabled by default)
- Shows what will be fixed before fixing
- Only updates articles with missing/invalid author
- Saves detailed log to JSON file

In [ ]:
# Correct author ID for all articles
CORRECT_AUTHOR_ID = "6ea5d061-1dff-454e-83a7-7b83d4546ed4"

print(f"Correct author ID: {CORRECT_AUTHOR_ID}")

In [ ]:
def find_articles_with_invalid_author():
    """Find all articles with null, empty, or invalid author data"""
    print("=" * 60)
    print("FINDING ARTICLES WITH INVALID AUTHOR DATA")
    print("=" * 60)
    print()
    
    invalid_articles = []
    
    for category_name, category_id in WA_CATEGORY_IDS.items():
        if not category_id:
            continue
        
        print(f"Checking {category_name}...")
        
        try:
            articles = wa_api.get_articles_by_category(category_id)
            
            for article in articles:
                author = article.get("author")
                
                # Check if author is null, empty, or has no/invalid ID
                needs_fix = False
                reason = ""
                
                if author is None:
                    needs_fix = True
                    reason = "author is null"
                elif not isinstance(author, dict):
                    needs_fix = True
                    reason = "author is not an object"
                elif not author.get("id"):
                    needs_fix = True
                    reason = "author has no id"
                elif author.get("id") != CORRECT_AUTHOR_ID:
                    needs_fix = True
                    reason = f"author id is {author.get('id')[:12]}... (incorrect)"
                
                if needs_fix:
                    invalid_articles.append({
                        "id": article.get("id"),
                        "title": article.get("title", "Untitled"),
                        "category": category_name,
                        "entityClass": article.get("entityClass", "Unknown"),
                        "reason": reason,
                        "current_author": author
                    })
            
            print(f"  Found {len([a for a in invalid_articles if a['category'] == category_name])} articles needing fix")
            
        except Exception as e:
            print(f"  Error checking {category_name}: {e}")
    
    print()
    print("=" * 60)
    print(f"Total articles with invalid author: {len(invalid_articles)}")
    print("=" * 60)
    
    # Show sample
    if invalid_articles:
        print()
        print("Sample of articles needing fix:")
        for i, article in enumerate(invalid_articles[:10], 1):
            print(f"{i:2d}. [{article['category']}] {article['title']}")
            print(f"    ID: {article['id']}")
            print(f"    Reason: {article['reason']}")
        
        if len(invalid_articles) > 10:
            print(f"\n... and {len(invalid_articles) - 10} more")
    
    return invalid_articles

# Find articles with invalid author
invalid_articles = find_articles_with_invalid_author()

In [ ]:
# ============================================
# FIX AUTHOR DATA
# ============================================

DRY_RUN = False  # Set to False to actually fix

print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No articles will be updated")
else:
    print("⚠️  LIVE MODE - Articles WILL be updated!")
print("=" * 60)
print()

if not invalid_articles:
    print("✓ No articles need fixing")
else:
    print(f"Will fix {len(invalid_articles)} articles\n")
    
    fixed_count = 0
    failed_count = 0
    fix_log = []
    
    for i, article in enumerate(invalid_articles, 1):
        article_id = article["id"]
        title = article["title"]
        
        print(f"[{i}/{len(invalid_articles)}] Fixing: {title}")
        print(f"  Reason: {article['reason']}")
        
        if DRY_RUN:
            print(f"  → DRY RUN: Would set author to {CORRECT_AUTHOR_ID}")
            fix_log.append({
                "title": title,
                "id": article_id,
                "category": article['category'],
                "reason": article['reason'],
                "status": "dry_run"
            })
        else:
            try:
                # Use raw PATCH request to fix author
                url = f"{WA_API_BASE}/article"
                params = {"id": article_id}
                
                # Minimal payload - only fix author
                payload = {
                    "author": {
                        "id": CORRECT_AUTHOR_ID,
                        "entityClass": "User"
                    }
                }
                
                response = session.patch(url, params=params, json=payload, timeout=30)
                response.raise_for_status()
                
                print(f"  ✓ Fixed successfully")
                fixed_count += 1
                fix_log.append({
                    "title": title,
                    "id": article_id,
                    "category": article['category'],
                    "reason": article['reason'],
                    "status": "fixed"
                })
                
                # Rate limiting
                import time
                time.sleep(0.3)
                
            except Exception as e:
                print(f"  ✗ Failed: {e}")
                failed_count += 1
                fix_log.append({
                    "title": title,
                    "id": article_id,
                    "category": article['category'],
                    "reason": article['reason'],
                    "status": "failed",
                    "error": str(e)
                })
    
    print()
    print("=" * 60)
    print("FIX SUMMARY")
    print("=" * 60)
    if DRY_RUN:
        print(f"DRY RUN: Would fix {len(invalid_articles)} articles")
        print("\nSet DRY_RUN = False to actually fix")
    else:
        print(f"✓ Fixed: {fixed_count}")
        print(f"✗ Failed: {failed_count}")
    print("=" * 60)
    
    # Save log
    import json
    from datetime import datetime
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"world_anvil/author_fix_{timestamp}.json"
    
    with open(log_file, 'w') as f:
        json.dump({
            "timestamp": timestamp,
            "dry_run": DRY_RUN,
            "correct_author_id": CORRECT_AUTHOR_ID,
            "total_articles": len(invalid_articles),
            "fixed": fixed_count,
            "failed": failed_count,
            "articles": fix_log
        }, f, indent=2)
    
    print(f"\n✓ Log saved to: {log_file}")